# Inference on FPGA of HLS4ML-projects with Vitis Unified

This notebook is optimized to test performance across multiple synthesized models of the same dataset. Meaning the dataset is common, but architecture may be wildly different. It may be run in vanilla python-environment. 

It's based on [fastmachinelearning/hls4ml-tutorial/part7b_deployment.ipynb](https://github.com/fastmachinelearning/hls4ml-tutorial/blob/main/part7b_deployment.ipynb), [Tanawin1701d/vitisUnifiedTutorial/part8b_testOnHw.ipynbxt](https://github.com/Tanawin1701d/vitisUnifiedTutorial/blob/main/part8b_testOnHw.ipynb) and earlier experimentation. For syntax, see driverclass in directory and [pynq-package documentation](https://pynq.readthedocs.io/en/v2.5.1/pynq_package/pynq.overlay.html).


## Load project

Vitis Unified copies the final files needed for inference to the subfolder in HLS4ML-projectdirectory `export/`. 
Copy the bitfile (.bit) and descriptor (.hwh) into its own directory under `DUT/` (Device Under Test) with explanatory name on the directory.


In [26]:
search_dir = 'DUT/'
bitfile_pattern = '*.bit'
hwh_pattern = '*.hwh'

x_test_path = '../testmodel_2_VitisUnifiedKV260/x_test.npy'
y_test_path = '../testmodel_2_VitisUnifiedKV260/y_test.npy'

prediction_path = 'predictions/'

In [27]:
from pathlib import Path
duts = []

print(f"Searching for models to test in {search_dir}")
for bitfile_path in Path(search_dir).rglob(bitfile_pattern):
    #print(path, parent_dir)
    parent_dir = bitfile_path.parent
    if parent_dir.rglob(hwh_pattern): # make sure hwh is there as well
        # https://docs.python.org/3/library/pathlib.html#pathlib.PurePath
        project_name = f"{bitfile_path.parts[-2]}" 
        print(f"\n\n%%%%%%% Model to test: {project_name} ({bitfile_path}) %%%%%%%%%%%\n")
        duts.append({
            'project_name' : project_name,
            'bitfile_path' : bitfile_path
        })

Searching for models to test in DUT/


%%%%%%% Model to test: Training_FixedHP_acc=0.7590_ebops=11634_VU_DA_bitfile (DUT/Training_FixedHP_acc=0.7590_ebops=11634_VU_DA_bitfile/system.bit) %%%%%%%%%%%



%%%%%%% Model to test: Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile (DUT/Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile/system.bit) %%%%%%%%%%%



%%%%%%% Model to test: Training_AdaptiveHP_acc=0.7512_ebops=2895_VU_DA_bitfile (DUT/Training_AdaptiveHP_acc=0.7512_ebops=2895_VU_DA_bitfile/system.bit) %%%%%%%%%%%



%%%%%%% Model to test: Training_AdaptiveHP_acc=0.7117_ebops=303_VU_latency_bitfile (DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_latency_bitfile/system.bit) %%%%%%%%%%%



%%%%%%% Model to test: Training_AdaptiveHP_acc=0.7117_ebops=303_VU_DA_bitfile (DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_DA_bitfile/system.bit) %%%%%%%%%%%



In [28]:
duts

[{'project_name': 'Training_FixedHP_acc=0.7590_ebops=11634_VU_DA_bitfile',
  'bitfile_path': PosixPath('DUT/Training_FixedHP_acc=0.7590_ebops=11634_VU_DA_bitfile/system.bit')},
 {'project_name': 'Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile',
  'bitfile_path': PosixPath('DUT/Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile/system.bit')},
 {'project_name': 'Training_AdaptiveHP_acc=0.7512_ebops=2895_VU_DA_bitfile',
  'bitfile_path': PosixPath('DUT/Training_AdaptiveHP_acc=0.7512_ebops=2895_VU_DA_bitfile/system.bit')},
 {'project_name': 'Training_AdaptiveHP_acc=0.7117_ebops=303_VU_latency_bitfile',
  'bitfile_path': PosixPath('DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_latency_bitfile/system.bit')},
 {'project_name': 'Training_AdaptiveHP_acc=0.7117_ebops=303_VU_DA_bitfile',
  'bitfile_path': PosixPath('DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_DA_bitfile/system.bit')}]

Load datasets for testing performance in inference

In [29]:
import numpy as np
# Load input from .npy file as global variables for use in methods later (lazy hehe)
x_test = np.load(x_test_path)
y_test = np.load(y_test_path)

In [35]:
# import the library/driver which is common for every VItis Unified synthesis
from axi_master_driver import NeuralNetworkOverlay

# TODO: test if this actually works
def test_dut(bitfile_path, project_name, prediction_path):
    # create the overlay object
    overlay = NeuralNetworkOverlay(bitfile_name=bitfile_path, x_shape=x_test.shape, y_shape=y_test.shape, dtype=x_test.dtype)
    
    # Do the prediction/run inference
    result = overlay.predict(x_test, debug=False, profile=True, encode=np.float32, decode=np.float32)
    y_dut = result[0]

    # Save results
    np.save(f"{prediction_path}/y_dut_{project_name}.npy",y_dut)

    # TODO: calculate performance

    return result # contains the output buffer, execution time and inference/s

In [36]:
def cal_accuracy(y_dut):
    y_pred = np.argmax(y_dut, axis=1)
    y_true = np.argmax(y_test, axis=1)
    return np.sum(y_pred == y_true) / len(y_true)

# test the function 
#y_dut = np.load('../testmodel_2_VitisUnifiedKV260/y_hardware.npy')
#acc = cal_accuracy(y_dut)
#print(f"accuracy of hardware inference: {acc}")

Run the actual inference

In [48]:
import os
prediction_path = 'predictions'
os.makedirs(prediction_path,exist_ok=True)

for dut in duts:
    print(f"\n\nLoading and testing {dut['project_name']}")
    print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%\n")
    
    # do inference
    result = test_dut(str(dut['bitfile_path']), dut['project_name'], prediction_path)
    y_dut = result[0] # retrieving the 
    #print(y_dut[:10])

    # calculate acc
    acc = cal_accuracy(y_dut)
    print(f"\naccuracy of hardware inference: {acc}")

    # make ROC of some, sammenlikne baseline mot quantized og så mot kjørt på FPGA
    #make_roc(y_test, y_dut, ["q", "g", ...]) # TODO: complete this

    # generate confusion matrix ?



Loading and testing Training_FixedHP_acc=0.7590_ebops=11634_VU_DA_bitfile
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

input gmem_in0_ptr_linput will be set to addr: 0x3bf00000 with elements: 2656000
output gmem_out0_ptr_layer5_out will be set to addr: 0x3aa00000 with elements: 830000
amount of queries will be set to: 166000 at address: 0x28
prepare your interrupt
global interrupt enable register
enable gie successful
ap_done interrupt enable register
enable ap_done interrupt successful
ap_done register clear
clear ap_done interrupt successful
----------------------
starting the accelerator
Accelerator execution time:                       1.262 ms
accelerator has finished
Processed elements: 166000, Execution time: 0.0013 seconds, Performance rate: 131561010.98 inferences/second

accuracy of hardware inference: 0.7590481927710844


Loading and testing Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

input gmem_in0_ptr_linput 